# Lab 5: AEP Dataset - Normalization, One-Hot Encoding & Cyclic Features

**University of Engineering and Technology Peshawar**

**Student Name:** Muhammad Ayub
**Registration No:** [Your Reg No]

**Date:** May 22, 2026

## Objective
To apply Min-Max Normalization, One-Hot Encoding, and Cyclic Encoding on the AEP hourly energy dataset after feature extraction.

## Table of Contents
1. [Imports](#imports)
2. [Load Data](#load)
3. [Data Split](#split)
4. [Normalization](#norm)
5. [One-Hot Encoding](#onehot)
6. [Cyclic Encoding](#cyclic)
7. [Combine & Save](#save)

## 1. Imports <a id='imports'></a>

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from timeseires.utils import t_v_t_split as sp

## 2. Load Dataset <a id='load'></a>

In [ ]:
df = pd.read_csv(r'C:\Users\M Ayub\Downloads\ML_LAB\5_features_extracted.csv', 
                 index_col='Datetime', parse_dates=['Datetime'])
df.head()

In [ ]:
df.info()

## 3. Train / Validation / Test Split <a id='split'></a>

In [ ]:
train_set, validation_set, test_set = sp.t_v_t(df, 70, 20, 10)

print(train_set.shape, validation_set.shape, test_set.shape)

## 4. Normalization (MinMaxScaler) <a id='norm'></a>

In [ ]:
train_load = train_set['aep'].values.reshape(-1, 1)
val_load   = validation_set['aep'].values.reshape(-1, 1)
test_load  = test_set['aep'].values.reshape(-1, 1)

scaler = MinMaxScaler()
scaler.fit(train_load)

scaled_train = scaler.transform(train_load)
scaled_val   = scaler.transform(val_load)
scaled_test  = scaler.transform(test_load)

pickle.dump(scaler, open('AEPscaler.pkl', 'wb'))

## 5. One-Hot Encoding <a id='onehot'></a>

In [ ]:
enc_holiday = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc_weekend = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

holiday_train = enc_holiday.fit_transform(train_set[['holiday']])
weekend_train = enc_weekend.fit_transform(train_set[['weekend']])

holiday_val = enc_holiday.transform(validation_set[['holiday']])
weekend_val = enc_weekend.transform(validation_set[['weekend']])

holiday_test = enc_holiday.transform(test_set[['holiday']])
weekend_test = enc_weekend.transform(test_set[['weekend']])

## 6. Cyclic Features <a id='cyclic'></a>

In [ ]:
def create_cyclic_features(df):
    cyclic_cols = ['month', 'day_of_week', 'hour', 'winter', 'spring', 'summer', 'fall', 'year_day']
    periods = [12, 6, 24, 4, 4, 4, 4, 365]
    cyclic = []
    for col, period in zip(cyclic_cols, periods):
        cyclic.append(np.sin(2 * np.pi * df[col].values / period))
        cyclic.append(np.cos(2 * np.pi * df[col].values / period))
    return np.column_stack(cyclic)

train_cyclic = create_cyclic_features(train_set)
val_cyclic   = create_cyclic_features(validation_set)
test_cyclic  = create_cyclic_features(test_set)

## 7. Combine Features & Save <a id='save'></a>

In [ ]:
train_final = np.hstack((scaled_train, holiday_train, weekend_train, train_cyclic))
val_final   = np.hstack((scaled_val, holiday_val, weekend_val, val_cyclic))
test_final  = np.hstack((scaled_test, holiday_test, weekend_test, test_cyclic))

print("Final Shapes:")
print(train_final.shape, val_final.shape, test_final.shape)

# Save as CSV
np.savetxt('7_AEP_train.csv', train_final, delimiter=',', comments='')
np.savetxt('8_AEP_validation.csv', val_final, delimiter=',', comments='')
np.savetxt('9_AEP_test.csv', test_final, delimiter=',', comments='')

print("✅ Files saved successfully!")

---
**Conclusion:**

The AEP dataset has been successfully processed with proper normalization, categorical encoding, and cyclic features for time series modeling.

**GitHub Link:** (https://github.com/prince4775/8th-Semester-ML-and-DL-Lab)
